# Alpamayo 1.5 — Inferência Interativa
Carrega o modelo **uma vez** e mantém na memória para testes com imagens e vídeos.

In [ ]:
import os, sys, glob, random
import torch
import pandas as pd
from PIL import Image
from IPython.display import display, Image as IPImage

BASE = '/data/wesleyferreiramaia/wokzone-alpamayo'
sys.path.insert(0, os.path.join(BASE, 'alpamayo1.5', 'src'))
os.environ['WANDB_DISABLED'] = 'true'
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'

CKPT    = f'{BASE}/checkpoints/sft_stage1_roadwork/checkpoint-18000'
A1_CKPT = f'{BASE}/models/Alpamayo-1.5-10B-A1-format'
VLM_PATH= f'{BASE}/models/Cosmos-Reason2-8B'

print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

In [ ]:
from omegaconf import OmegaConf
from safetensors.torch import load_file
from transformers import AutoProcessor
from alpamayo1_5_sft.models.sft_base_model import TrainableReasoningVLA

print('Carregando modelo base...')
model = TrainableReasoningVLA.from_alpamayo_checkpoint(
    checkpoint_path=A1_CKPT,
    vlm_name_or_path=VLM_PATH,
)
print('Carregando pesos do SFT...')
shards = sorted(glob.glob(os.path.join(CKPT, 'model-*.safetensors')))
sd = {}
for f in shards:
    sd.update(load_file(f, device='cpu'))
missing, _ = model.load_state_dict(sd, strict=False)
print(f'  missing={len(missing)}')

model = model.to(torch.bfloat16).cuda().eval()
processor = AutoProcessor.from_pretrained(VLM_PATH, trust_remote_code=True)
print('Modelo pronto!')

In [ ]:
def infer(image_or_path, question, max_new_tokens=256):
    """Roda inferência dado uma imagem (PIL ou path) e uma pergunta."""
    if isinstance(image_or_path, str):
        image = Image.open(image_or_path).convert('RGB')
        display(image.resize((640, 360)))
    else:
        image = image_or_path

    messages = [{
        'role': 'user',
        'content': [
            {'type': 'image', 'image': image},
            {'type': 'text',  'text': question},
        ]
    }]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[image], return_tensors='pt', padding=True)
    inputs = {k: v.cuda() if isinstance(v, torch.Tensor) else v for k, v in inputs.items()}

    with torch.no_grad():
        with torch.autocast('cuda', dtype=torch.bfloat16):
            generated = model.vlm.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=processor.tokenizer.eos_token_id,
            )
    n_in = inputs['input_ids'].shape[1]
    answer = processor.tokenizer.decode(generated[0][n_in:], skip_special_tokens=True).strip()
    print(f'Q: {question}')
    print(f'\nA: {answer}')
    return answer

## Teste com imagens do val set

In [ ]:
DATA_ROOT = f'{BASE}/data/roadwork/lingoqa_roadwork'
df = pd.read_parquet(f'{DATA_ROOT}/val.parquet')
print(f'{len(df)} exemplos no val set')
df.head(3)

In [ ]:
# Pegar um exemplo aleatório do val
row = df.sample(1, random_state=42).iloc[0]
img_path = os.path.join(DATA_ROOT, row['images'][0])

print(f'Segmento: {row["segment_id"]}')
print(f'Pergunta: {row["question"]}')
print(f'\nLabel esperado:\n{row["answer"]}')
print('\n--- Resposta do modelo ---')
infer(img_path, row['question'])

## Teste com imagem própria

In [ ]:
# Substitua pelo path de qualquer imagem
# infer('/path/para/sua/imagem.jpg', 'Describe the work zone hazards in this scene.')

## Teste com frame de vídeo

In [ ]:
import cv2

def infer_video(video_path, question, frame_num=0):
    """Extrai um frame do vídeo e roda inferência."""
    cap = cv2.VideoCapture(video_path)
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_num)
    ret, frame = cap.read()
    cap.release()
    if not ret:
        print('Erro ao ler frame')
        return
    image = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    print(f'Frame {frame_num} do vídeo: {os.path.basename(video_path)}')
    return infer(image, question)

# Exemplo:
# videos = glob.glob(f'{BASE}/data/roadwork/videos/*.mp4')
# infer_video(videos[0], 'What work zone hazards are present?', frame_num=30)